In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Make charts a consistent, readable size and style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

In [12]:
from google.colab import files
uploaded = files.upload()

Saving hypertension_dataset_5000 (1).csv to hypertension_dataset_5000 (1) (2).csv


In [13]:
# Using the exact filename i uploaded
df = pd.read_csv("hypertension_dataset_5000 (1).csv")

# Keep an untouched copy so i can always compare before/after cleaning
df_raw = df.copy()

# First look: 5 random rows
df.sample(5, random_state=42)

,Age,Gender,BMI,Waist_Circumference_cm,Salt_Intake,Physical_Activity,Alcohol_Consumption,Smoking_Status,Stress_Level,Sleep_Hours,Family_History_HTN,Creatinine_mg_dL,Fasting_Glucose_mg_dL,Cholesterol_mg_dL,Systolic_BP_mmHg,Diastolic_BP_mmHg,BP_Category,Hypertension
1501,89.2,Female,22.7,83.9,Very High,Light,NaN,Never,Moderate,7.2,Yes,0.84,90.1,271.0,132,84,Stage 1 Hypertension,Yes
2586,70.8,Male,28.2,99.1,Low,Light,Occasional,Never,Low,6.1,No,0.92,77.8,NaN,106,76,Normal,No
2653,53.3,Female,28.1,81.5,Normal,Sedentary,Moderate,Former,High,4.8,Yes,0.80,94.9,191.4,143,97,Stage 2 Hypertension,Yes
1055,26.8,Male,24.2,86.4,Normal,Active,Occasional,Never,Very High,6.5,No,1.04,85.4,161.8,100,73,Normal,No
705,81.9,Male,26.4,100.0,Low,Moderate,NaN,Never,Moderate,7.0,Yes,0.95,94.5,173.4,111,75,Normal,No


In [14]:
# shape = (rows, columns)
print("Rows, columns:", df.shape)

# info() shows each column's type and how many non-null values it has
df.info()

Rows, columns: (5000, 18)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Age                     5000 non-null   float64
 1   Gender                  5000 non-null   object 
 2   BMI                     5000 non-null   float64
 3   Waist_Circumference_cm  4846 non-null   float64
 4   Salt_Intake             4939 non-null   object 
 5   Physical_Activity       5000 non-null   object 
 6   Alcohol_Consumption     3632 non-null   object 
 7   Smoking_Status          5000 non-null   object 
 8   Stress_Level            4909 non-null   object 
 9   Sleep_Hours             4837 non-null   float64
 10  Family_History_HTN      5000 non-null   object 
 11  Creatinine_mg_dL        4867 non-null   float64
 12  Fasting_Glucose_mg_dL   4821 non-null   float64
 13  Cholesterol_mg_dL       4841 non-null   float64
 14  Systolic_BP_mm

In [15]:
def missing_report(data):
    """Return a table of missing counts and percentages, largest first."""
    counts = data.isna().sum()
    pct = (counts / len(data) * 100).round(1)
    report = pd.DataFrame({"missing": counts, "percent": pct})
    # Show only columns that actually have missing values
    return report[report["missing"] > 0].sort_values("missing", ascending=False)

missing_report(df)

,missing,percent
Alcohol_Consumption,1368,27.4
Fasting_Glucose_mg_dL,179,3.6
Sleep_Hours,163,3.3
Cholesterol_mg_dL,159,3.2
Waist_Circumference_cm,154,3.1
Creatinine_mg_dL,133,2.7
Stress_Level,91,1.8
Salt_Intake,61,1.2


In [16]:
# Count fully duplicated rows
print("Duplicate rows:", df.duplicated().sum())

# Summary statistics for numeric columns: look at min and max for
# impossible or extreme values (e.g. negative age, BMI of 5)
df.describe().T

Duplicate rows: 0


,count,mean,std,min,25%,50%,75%,max
Age,5000.0,48.648320,16.908037,18.20,34.575,47.40,61.60,89.20
BMI,5000.0,27.876320,4.254628,16.20,24.900,27.70,30.80,43.50
Waist_Circumference_cm,4846.0,89.417623,11.410790,54.00,81.500,89.10,97.10,126.40
Sleep_Hours,4837.0,6.850444,1.042016,3.50,6.200,6.90,7.50,10.50
Creatinine_mg_dL,4867.0,0.942063,0.171618,0.51,0.820,0.93,1.05,1.68
Fasting_Glucose_mg_dL,4821.0,93.301058,12.753358,62.10,84.500,92.20,100.90,173.40
Cholesterol_mg_dL,4841.0,187.219851,30.546704,105.00,165.600,187.30,207.60,288.30
Systolic_BP_mmHg,5000.0,116.826600,16.385269,88.00,104.000,115.00,128.00,181.00
Diastolic_BP_mmHg,5000.0,74.259600,9.483849,50.00,67.000,74.00,81.00,110.00


In [17]:
# Identify text columns, then list every category and its count.
# dropna=False keeps missing values visible instead of hiding them.
cat_cols = df.select_dtypes(include=["object", "string"]).columns

for col in cat_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))


--- Gender ---
Gender
Female    2581
Male      2419
Name: count, dtype: int64

--- Salt_Intake ---
Salt_Intake
Normal       1696
High         1597
Very High    1011
Low           635
NaN            61
Name: count, dtype: int64

--- Physical_Activity ---
Physical_Activity
Light        1678
Sedentary    1327
Moderate     1258
Active        737
Name: count, dtype: int64

--- Alcohol_Consumption ---
Alcohol_Consumption
Occasional    1910
NaN           1368
Moderate      1231
Heavy          491
Name: count, dtype: int64

--- Smoking_Status ---
Smoking_Status
Never      2805
Former     1440
Current     755
Name: count, dtype: int64

--- Stress_Level ---
Stress_Level
Moderate     1844
High         1410
Low           844
Very High     811
NaN            91
Name: count, dtype: int64

--- Family_History_HTN ---
Family_History_HTN
No     3218
Yes    1782
Name: count, dtype: int64

--- BP_Category ---
BP_Category
Normal                  2713
Stage 1 Hypertension    1147
Stage 2 Hypertension     6

In [18]:
# Does BP_Category perfectly predict Hypertension?
# A crosstab counts how many patients fall in each combination.
print(pd.crosstab(df["BP_Category"], df["Hypertension"]))

# Range of BP values within each target group
print(df.groupby("Hypertension")[["Systolic_BP_mmHg", "Diastolic_BP_mmHg"]]
        .agg(["min", "max"]))

Hypertension            No   Yes
BP_Category                     
Elevated               537     0
Normal                2713     0
Stage 1 Hypertension     0  1147
Stage 2 Hypertension     0   603
             Systolic_BP_mmHg      Diastolic_BP_mmHg     
                          min  max               min  max
Hypertension                                             
No                         88  129                50   79
Yes                       105  181                59  110


In [19]:
# Keeping my decisions in one place
NUMERIC_MISSING = ["Fasting_Glucose_mg_dL", "Sleep_Hours", "Cholesterol_mg_dL",
                   "Waist_Circumference_cm", "Creatinine_mg_dL"]
CATEGORICAL_MISSING = ["Stress_Level", "Salt_Intake"]


def fill_alcohol_none(data):
    """Alcohol has no 'None' category, so blanks most likely mean 'does not
    drink'. Recode them rather than guessing a drinker level."""
    data = data.copy()                      # never modify the caller's data
    data["Alcohol_Consumption"] = data["Alcohol_Consumption"].fillna("None")
    return data


def impute_numeric_median(data, columns):
    """Fill numeric gaps with the median. Median is robust to the skew and
    outliers seen in glucose and creatinine, and each gap is under 4%."""
    data = data.copy()
    for col in columns:
        data[col] = data[col].fillna(data[col].median())
    return data


def impute_categorical_mode(data, columns):
    """Fill categorical gaps with the most common category (both gaps <2%)."""
    data = data.copy()
    for col in columns:
        data[col] = data[col].fillna(data[col].mode()[0])
    return data

In [20]:
# Does having a missing value relate to hypertension? If the rate is very
# different, the missingness itself carries information and needs care.
for col in ["Alcohol_Consumption", "Fasting_Glucose_mg_dL", "Sleep_Hours"]:
    is_missing = df_raw[col].isna()
    rate = (df_raw["Hypertension"].eq("Yes")
            .groupby(is_missing).mean().round(3))
    print(f"{col}: HTN rate when present vs missing\n{rate}\n")

Alcohol_Consumption: HTN rate when present vs missing
Alcohol_Consumption
False    0.371
True     0.293
Name: Hypertension, dtype: float64

Fasting_Glucose_mg_dL: HTN rate when present vs missing
Fasting_Glucose_mg_dL
False    0.353
True     0.268
Name: Hypertension, dtype: float64

Sleep_Hours: HTN rate when present vs missing
Sleep_Hours
False    0.349
True     0.393
Name: Hypertension, dtype: float64



In [21]:
df = df_raw.copy()                                   # start from raw each time
df = fill_alcohol_none(df)
df = impute_numeric_median(df, NUMERIC_MISSING)
df = impute_categorical_mode(df, CATEGORICAL_MISSING)

print("Missing values left:", df.isna().sum().sum())   # should be 0
print("Duplicates:", df.duplicated().sum())            # 0, so nothing to drop

# Impact check: did imputing distort the distributions?
# Means/std should barely move.
before = df_raw[NUMERIC_MISSING].describe().loc[["mean", "std"]]
after = df[NUMERIC_MISSING].describe().loc[["mean", "std"]]
print(before.round(2)); print(after.round(2))

Missing values left: 0
Duplicates: 0
      Fasting_Glucose_mg_dL  Sleep_Hours  Cholesterol_mg_dL  \
mean                  93.30         6.85             187.22   
std                   12.75         1.04              30.55   

      Waist_Circumference_cm  Creatinine_mg_dL  
mean                   89.42              0.94  
std                    11.41              0.17  
      Fasting_Glucose_mg_dL  Sleep_Hours  Cholesterol_mg_dL  \
mean                  93.26         6.85             187.22   
std                   12.52         1.02              30.06   

      Waist_Circumference_cm  Creatinine_mg_dL  
mean                   89.41              0.94  
std                    11.23              0.17  


In [22]:
# ACC/AHA-style rule: hypertension if systolic >= 130 OR diastolic >= 80
rule = (df["Systolic_BP_mmHg"] >= 130) | (df["Diastolic_BP_mmHg"] >= 80)
rule_label = rule.map({True: "Yes", False: "No"})

# How often does the rule reproduce the dataset's label?
print("Agreement:", (rule_label == df["Hypertension"]).mean())
print(pd.crosstab(rule_label, df["Hypertension"]))

Agreement: 1.0
Hypertension    No   Yes
row_0                   
No            3250     0
Yes              0  1750


In [23]:
def add_bp_features(data):
    """Features that describe BP without just repeating the label."""
    data = data.copy()
    sys_, dia = data["Systolic_BP_mmHg"], data["Diastolic_BP_mmHg"]
    # Pulse pressure: gap between systolic and diastolic. High values suggest
    # stiffer arteries and are more typical in older patients.
    data["Pulse_Pressure"] = sys_ - dia
    # Mean arterial pressure: average pressure driving blood to organs.
    data["MAP"] = (dia + (sys_ - dia) / 3).round(1)
    return data


def add_clinical_groups(data):
    """Turn raw numbers into groups clinicians already use."""
    data = data.copy()
    # BMI categories (WHO): <18.5, 18.5-24.9, 25-29.9, 30+
    data["BMI_Category"] = pd.cut(
        data["BMI"], bins=[0, 18.5, 25, 30, np.inf], right=False,
        labels=["Underweight", "Healthy", "Overweight", "Obese"])
    # Age bands for comparing life stages
    data["Age_Group"] = pd.cut(
        data["Age"], bins=[0, 30, 45, 60, 75, np.inf], right=False,
        labels=["18-29", "30-44", "45-59", "60-74", "75+"])
    # Waist risk differs by sex (NHS: men >102 cm, women >88 cm = high risk)
    cutoff = np.where(data["Gender"] == "Male", 102, 88)
    data["High_Waist_Risk"] = np.where(
        data["Waist_Circumference_cm"] > cutoff, "Yes", "No")
    # Glucose (mg/dL): <100 normal, 100-125 prediabetic range, 126+ diabetic range
    data["Glucose_Category"] = pd.cut(
        data["Fasting_Glucose_mg_dL"], bins=[0, 100, 126, np.inf],
        right=False, labels=["Normal", "Prediabetic range", "Diabetic range"])
    return data


df = add_bp_features(df)
df = add_clinical_groups(df)
df[["Pulse_Pressure", "MAP", "BMI_Category", "Age_Group",
    "High_Waist_Risk", "Glucose_Category"]].head()

,Pulse_Pressure,MAP,BMI_Category,Age_Group,High_Waist_Risk,Glucose_Category
0,54,83.0,Healthy,75+,No,Normal
1,51,98.0,Overweight,60-74,No,Normal
2,44,101.7,Healthy,45-59,No,Normal
3,39,101.0,Overweight,45-59,No,Normal
4,42,92.0,Overweight,30-44,No,Normal


In [24]:
# Ordered categories get ordered numbers, since "Very High" > "Low" matters.
ORDER_MAPS = {
    "Salt_Intake":         {"Low": 0, "Normal": 1, "High": 2, "Very High": 3},
    "Physical_Activity":   {"Active": 0, "Moderate": 1, "Light": 2, "Sedentary": 3},
    "Alcohol_Consumption": {"None": 0, "Occasional": 1, "Moderate": 2, "Heavy": 3},
    "Stress_Level":        {"Low": 0, "Moderate": 1, "High": 2, "Very High": 3},
    "Smoking_Status":      {"Never": 0, "Former": 1, "Current": 2},
}
# Two-category columns become 0/1.
BINARY_MAPS = {"Gender": {"Female": 0, "Male": 1},
               "Family_History_HTN": {"No": 0, "Yes": 1},
               "Hypertension": {"No": 0, "Yes": 1}}


def encode_columns(data):
    """Add *_Enc columns; keep the readable originals for charts."""
    data = data.copy()
    for col, mapping in {**ORDER_MAPS, **BINARY_MAPS}.items():
        data[col + "_Enc"] = data[col].map(mapping)
    # Simple lifestyle score: higher = more risk factors (equal weights)
    data["Lifestyle_Risk_Score"] = data[[c + "_Enc" for c in ORDER_MAPS]].sum(axis=1)
    return data


df = encode_columns(df)
print(df.filter(like="_Enc").isna().sum().sum(), "unmapped values")   # expect 0
df.to_csv("hypertension_clean.csv", index=False)   # reuse in the Streamlit app

0 unmapped values
